# Amazon SageMaker AI: controlled cloud lab

Follow [the full cloud lesson](../../docs/10-aws-lifecycle.md) and its setup lesson first.
Cloud actions allocate billable resources. **Do not use Run All for a live lab**:
run one stage, inspect its output, and fill in the resulting identifiers.

`RUN_CLOUD` defaults to `False`. The verification script also skips every cell
tagged `cloud`. Install the `aws` extra and select this project's kernel
before enabling cloud execution. Credentials come from your terminal login.


In [ ]:
import os
from iris_mlops.paths import PROJECT_DIR
import subprocess
import sys
from pathlib import Path

RUN_CLOUD = False

def step(module, *arguments):
    command = [sys.executable, "-m", "iris_mlops." + module, *map(str, arguments)]
    print(" ".join(command))
    if RUN_CLOUD:
        subprocess.run(command, check=True, cwd=PROJECT_DIR)
    else:
        print("Skipped: enable RUN_CLOUD only after completing setup and reviewing this step")

print("Cloud execution enabled:", RUN_CLOUD)


In [ ]:
IMAGE = os.getenv("AWS_IMAGE", "replace-with-ecr-uri@sha256:digest")
JOB = "iris-course-train-v1"
RUN_ID = "replace-after-import"
BUNDLE = Path("releases") / RUN_ID
PACKAGE = "replace-with-model-package-arn"
STAGING = "iris-course-stage-v1"
PRODUCTION = "iris-course-prod"


## 1. Train and download

Inspect the training job and CloudWatch logs. Downloads contain a candidate,
not an approved release. Use new names/directories for repeated runs.


In [ ]:
step("aws_cloud", "train", "--name", JOB, "--image", IMAGE)
step("aws_cloud", "download", "--name", JOB, "--output", "downloads/aws-v1")


## 2. Import and review

Copy the imported ID into `RUN_ID` and update `BUNDLE`. Evaluate before promotion.


In [ ]:
step("workflow", "import", "downloads/aws-v1")


In [ ]:
step("workflow", "evaluate", RUN_ID)


In [ ]:
step("workflow", "promote", RUN_ID)
step("workflow", "export", RUN_ID, "--output", BUNDLE)
step("aws_cloud", "register", "--bundle", BUNDLE, "--image", IMAGE)


## 3. Native registry approval

Registration prints the package ARN. Set `PACKAGE` above, inspect its metadata
in SageMaker Model Registry, and approve this exact release.


In [ ]:
step("aws_cloud", "approve", "--package", PACKAGE, "--bundle", BUNDLE)


## 4. Stage and promote

The package must be Approved. Production reuses the exact staging configuration.
Staging and production allocate separate compute while both exist.


In [ ]:
step("aws_cloud", "deploy", "--package", PACKAGE, "--name", STAGING)
step("aws_cloud", "smoke", "--endpoint", STAGING)


In [ ]:
step("aws_cloud", "promote", "--staging", STAGING, "--production", PRODUCTION)
step("aws_cloud", "status", "--endpoint", PRODUCTION)
step("aws_cloud", "smoke", "--endpoint", PRODUCTION)


## 5. Monitor and recover

Use CloudWatch and S3 capture as described in the lesson. After deploying a v2
release, restore the previous config. Verify the returned version after rollback.

Follow [lesson 10, step 4](../../docs/10-aws-lifecycle.md#4-update-and-recover) for each v2 command.
Keep the v1 run ID and endpoint config; after cloud rollback, restore the local
pointer with `uv run iris rollback RUN_ID_V1`.


In [ ]:
# After releasing v2, uncomment to recover v1:
# step("aws_cloud", "rollback", "--endpoint", PRODUCTION, "--config", "iris-course-stage-v1")


## 6. Cleanup

Use `uv run --extra aws iris-cleanup-aws` exactly as described in the lesson, listing every resource
you created. Deleting only endpoints leaves registry, image, storage and logs.
The optional purge permanently deletes the dedicated stack's retained storage.


In [ ]:
# Example for a lab that created only v1 (review before uncommenting):
# step("aws_cleanup", "--stack", "iris-course", "--endpoint", STAGING,
#      "--endpoint", PRODUCTION, "--config", STAGING, "--model", STAGING,
#      "--purge-storage")
